# Our test generation v1.1 — Kaggle / self-contained captioning

Sinh caption **source, target, adversarial** từ output đã giải nén của `test_attack_v1.ipynb`. Notebook tự chứa code, **không tải Git repo, không import module trong repo và không đọc cấu hình YAML bên ngoài**. Chọn model và sửa tham số trực tiếp tại Cell 1; sửa phiên bản package tại Cell 2.

**Cách chạy:** Add Input output attack; chỉnh `INPUT_DIR`, `MODEL`, `NUM_SAMPLES`. Bật Internet. GPT-5 mini dùng Secret `OPENAI_API_KEY`; Gemini dùng `GEMINI_API_KEY`. LLaVA/InternVL/Qwen cần GPU, không cần API key. Mặc định `RUN_CAPTION=False` để kiểm tra input; đặt `True` và chạy lại các cell để thực nghiệm. Chạy `NUM_SAMPLES=1` trước; `None` chạy toàn bộ manifest. Nếu thay package đã import, restart session rồi chạy lại notebook.

**Nguồn code để đối chiếu:**

- Cell 4: `ImageDescriptionGenerator` và các helper được chép nguyên văn từ `src/FOA-Attack/blackbox_text_generation.py` và `utils.py`. Class `KaggleRuntime` chỉ giữ nguyên phương thức `secret`, không mang theo logic đọc config của repo.
- Cell 5: chép nguyên văn `VersionedImageDescriptionGenerator` trong repo hiện tại. Đây là adapter bổ sung hỗ trợ 5 model; **paper FOA gốc không có đủ cả 5 model này**. Class paper gốc chỉ hỗ trợ GPT-4o/Claude/Gemini và được giữ ở Cell 4 để đối chiếu. Chạy thực tế bằng class ở Cell 5.
- Cell 6–8: nối manifest, ghi artifacts và preview ở mức top-level; không thêm hàm sinh caption mới.

Các giá trị mặc định tại Cell 1 giữ từ protocol v1 trước đó. Notebook không khóa prompt/model/token limit bằng các assert so với YAML, nên bạn có thể sửa để ablation. Những giá trị **hardcode** trong implementation nguyên bản nằm ngay ở **Cell 5**: retry `min=1, max=30, attempts=3`, OpenAI SDK `max_retries=3`, Gemini `temperature=0, thinking_budget=0`, local `do_sample=False`, GPU 0, dtype tự chọn theo GPU, NF4 double quantization, attention `eager`, `trust_remote_code=False`, OpenAI image detail `high`, `store=False`. Muốn đổi các giá trị này, sửa trực tiếp Cell 5; không có config field giả mà class không đọc.

`max_output_tokens` dùng cho API, `max_new_tokens` dùng cho local; `reasoning_effort` dùng cho OpenAI; `revision`/`quantization` dùng cho local. Đổi seed ở Cell 1; class gốc áp dụng seed khi tải local model. Local implementation chỉ hỗ trợ `quantization="nf4"` hoặc `"none"`.

Mỗi run xuất caption JSONL, cấu hình đã dùng, fingerprint cấu hình + code thực thi, metadata, snapshot các cell implementation và ZIP không kèm ảnh. Giữ output attack riêng nếu cần preview lại. `captions.jsonl` có `sample_id`, `caption_model`, `caption_config_sha256`, `captions.target` và `captions.adversarial` để đưa vào `our_gpt_evaluation_v1.ipynb`. Output `config_resolved.yaml` chỉ là bản ghi kết quả, không phải input cấu hình. Captioning và GPT evaluation nên chạy trong session riêng do phiên bản SDK khác nhau.


In [ ]:
# Cell 1 — Cấu hình thực nghiệm; chỉnh toàn bộ tham số đọc qua cfg tại đây.
from pathlib import Path

MODEL = "LLaVA-1.5-7B"
INPUT_DIR = Path("/kaggle/input/foa-attack-output")
# Cùng session attack: INPUT_DIR = Path("/kaggle/working/foa_attack_output")
OUTPUT_DIR = Path("/kaggle/working/our_text_generation_outputs")
NUM_SAMPLES = 1          # None: toàn bộ manifest; 1: smoke test.
RUN_CAPTION = False
SEED = 2023
EXPERIMENT_NAME = "our_test_generation_v1_1"

# Các bộ tham số hoàn chỉnh; chọn bằng MODEL. Chỉnh prompt/token limits/model_id tại đây.
MODEL_SETTINGS = {
    "GPT-5 mini": {
        "provider": "openai",
        "model_id": "gpt-5-mini-2025-08-07",
        "prompt": "Describe this image in one concise sentence, no longer than 20 words.",
        "timeout": 90,
        "reasoning_effort": "minimal",
        "max_output_tokens": 2048,
        "max_new_tokens": 128,
        "revision": "main",
        "quantization": "none",
    },
    "Gemini 2.5 Flash": {
        "provider": "gemini",
        "model_id": "gemini-2.5-flash",
        "prompt": "Describe this image in one concise sentence, no longer than 20 words.",
        "timeout": 90,
        "reasoning_effort": "minimal",
        "max_output_tokens": 2048,
        "max_new_tokens": 128,
        "revision": "main",
        "quantization": "none",
    },
    "LLaVA-1.5-7B": {
        "provider": "huggingface",
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "prompt": "Describe this image in one concise sentence, no longer than 20 words.",
        "timeout": 90,
        "reasoning_effort": "minimal",
        "max_output_tokens": 2048,
        "max_new_tokens": 128,
        "revision": "main",
        "quantization": "nf4",
    },
    "InternVL3-8B": {
        "provider": "huggingface",
        "model_id": "OpenGVLab/InternVL3-8B-hf",
        "prompt": "Describe this image in one concise sentence, no longer than 20 words.",
        "timeout": 90,
        "reasoning_effort": "minimal",
        "max_output_tokens": 2048,
        "max_new_tokens": 128,
        "revision": "main",
        "quantization": "nf4",
    },
    "Qwen3-VL-8B": {
        "provider": "huggingface",
        "model_id": "Qwen/Qwen3-VL-8B-Instruct",
        "prompt": "Describe this image in one concise sentence, no longer than 20 words.",
        "timeout": 90,
        "reasoning_effort": "minimal",
        "max_output_tokens": 2048,
        "max_new_tokens": 128,
        "revision": "main",
        "quantization": "nf4",
    },
}

if MODEL not in MODEL_SETTINGS:
    raise ValueError(f"Chọn MODEL trong {tuple(MODEL_SETTINGS)}")
LOCAL_MODEL = MODEL_SETTINGS[MODEL]["provider"] == "huggingface"
if NUM_SAMPLES is not None and (type(NUM_SAMPLES) is not int or NUM_SAMPLES <= 0):
    raise ValueError("NUM_SAMPLES cần là số nguyên dương hoặc None.")
if type(RUN_CAPTION) is not bool:
    raise ValueError("RUN_CAPTION cần là True hoặc False.")
if type(SEED) is not int or SEED < 0:
    raise ValueError("SEED cần là số nguyên không âm.")


In [ ]:
# Cell 2 — Chỉnh dependencies tại đây; giữ Torch/Torchvision của Kaggle.
# Pins mặc định theo caption protocol v1 trong repo trước đây; không đọc file config.
import subprocess
import sys

COMMON_PACKAGES = [
    "omegaconf==2.3.0", "PyYAML==6.0.2", "tenacity==9.0.0",
    "openai==1.109.1", "google-genai==1.41.0", "anthropic==0.45.2",
    "tqdm==4.67.1",
]
LOCAL_PACKAGES = [
    "transformers==4.57.6", "accelerate==1.10.1",
    "bitsandbytes==0.48.1", "sentencepiece==0.2.1",
]
PACKAGES = COMMON_PACKAGES + (LOCAL_PACKAGES if LOCAL_MODEL else [])
subprocess.run([sys.executable, "-m", "pip", "install", *PACKAGES], check=True)
print("Dependencies installed. Restart session nếu package cũ đã import trước khi đổi phiên bản.")


In [ ]:
# Cell 3 — Tạo cfg hoàn toàn từ Cell 1; OmegaConf chỉ bọc dict, không đọc YAML.
import os
import json
import hashlib
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "experiment": {"name": EXPERIMENT_NAME, "version": "v1.1", "seed": SEED},
    "caption": MODEL_SETTINGS[MODEL],
})
if cfg.caption.provider not in {"openai", "gemini", "huggingface"}:
    raise ValueError("provider cần là openai, gemini hoặc huggingface.")
if not isinstance(cfg.caption.model_id, str) or not cfg.caption.model_id.strip():
    raise ValueError("model_id không được rỗng.")
if not isinstance(cfg.caption.prompt, str) or not cfg.caption.prompt.strip():
    raise ValueError("prompt không được rỗng.")
if cfg.caption.timeout <= 0 or cfg.caption.max_output_tokens <= 0 or cfg.caption.max_new_tokens <= 0:
    raise ValueError("timeout và token limits cần dương.")
if LOCAL_MODEL and cfg.caption.quantization not in {"nf4", "none"}:
    raise ValueError("Class local nguyên bản hỗ trợ quantization=nf4 hoặc none.")

os.environ.update({
    "HF_HOME": "/kaggle/working/our_caption_hf_cache",
    "TOKENIZERS_PARALLELISM": "false",
})
if LOCAL_MODEL:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Bật GPU trong Kaggle Settings.")
    print("GPU 0:", torch.cuda.get_device_name(0))

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Add Input output attack và chỉnh INPUT_DIR: {INPUT_DIR}")
manifest_candidates = sorted(INPUT_DIR.rglob("manifest.jsonl"))
if len(manifest_candidates) != 1:
    raise ValueError(f"Cần đúng một manifest.jsonl; tìm thấy {len(manifest_candidates)}. Chọn thư mục một attack run.")
MANIFEST_PATH = manifest_candidates[0]
INPUT_ROOT = MANIFEST_PATH.parent.resolve()
print("Model:", MODEL, "| Run:", RUN_CAPTION)
print("Manifest:", MANIFEST_PATH)
print(OmegaConf.to_yaml(cfg, resolve=True))


In [ ]:
# Cell 4 — Code legacy và helpers chép nguyên văn; không import từ repo.
import base64
import yaml
from typing import Dict, Any, List, Tuple
from PIL import Image
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import OpenAI
from google import genai
import anthropic

def load_api_keys() -> Dict[str, str]:
    """Load API keys from the api_keys file.
    
    Returns:
        Dict[str, str]: Dictionary containing API keys for different models
        
    Raises:
        FileNotFoundError: If no api_keys file is found
    """
    for ext in ['yaml', 'yml', 'json']:
        file_path = f'api_keys.{ext}'
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                if ext in ['yaml', 'yml']:
                    return yaml.safe_load(f)
                else:
                    return json.load(f)
    
    raise FileNotFoundError(
        "API keys file not found. Please create api_keys.yaml, api_keys.yml, or api_keys.json "
        "in the root directory with your API keys."
    )

def get_api_key(model_name: str) -> str:
    """Get API key for specified model.
    
    Args:
        model_name: Name of the model to get API key for
        
    Returns:
        str: API key for the specified model
        
    Raises:
        KeyError: If API key for model is not found
    """
    api_keys = load_api_keys()
    if model_name not in api_keys:
        raise KeyError(
            f"API key for {model_name} not found in api_keys file. "
            f"Available models: {list(api_keys.keys())}"
        )
    return api_keys[model_name]

def encode_image(image_path: str) -> str:
    """Encode image file to base64 string.
    
    Args:
        image_path: Path to image file
        
    Returns:
        str: Base64 encoded image string
    """
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

class KaggleRuntime:
    @staticmethod
    def secret(name):
        value = os.environ.get(name)
        if value:
            return value
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            raise RuntimeError(f'Set environment variable or enable Kaggle Secret {name}.') from None
        if not value:
            raise RuntimeError(f'Empty secret: {name}')
        return value


def setup_gemini(api_key: str):
    return genai.Client(api_key=api_key)

def setup_claude(api_key: str):
    return anthropic.Anthropic(api_key=api_key)

def setup_gpt4o(api_key: str):
    return OpenAI(
        api_key="api_key",
    )

def get_media_type(image_path: str) -> str:
    """Get the correct media type based on file extension."""
    ext = os.path.splitext(image_path)[1].lower()
    if ext in [".jpg", ".jpeg", ".jpeg"]:
        return "image/jpeg"
    elif ext == ".png":
        return "image/png"
    else:
        raise ValueError(f"Unsupported image extension: {ext}")

class ImageDescriptionGenerator:
    def __init__(self, model_name: str):
        self.model_name = model_name
        # Get API key for the model
        api_key = get_api_key(model_name)

        if model_name == "gemini":
            self.client = setup_gemini(api_key)
        elif model_name == "claude":
            self.client = setup_claude(api_key)
        elif model_name == "gpt4o":
            self.client = setup_gpt4o(api_key)
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def generate_description(self, image_path: str) -> str:
        if self.model_name == "gemini":
            return self._generate_gemini(image_path)
        elif self.model_name == "claude":
            return self._generate_claude(image_path)
        elif self.model_name == "gpt4o":
            return self._generate_gpt4o(image_path)

    @retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
    def _generate_gemini(self, image_path: str) -> str:
        image = Image.open(image_path)
        response = self.client.models.generate_content(
            model="gemini-2.0-flash",
            contents=["Describe this image, no longer than 25 words.", image],
        )
        return response.text.strip()

    @retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
    def _generate_claude(self, image_path: str) -> str:
        base64_image = encode_image(image_path)
        media_type = get_media_type(image_path)
        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Describe this image in one concise sentence, no longer than 20 words.",
                        },
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": media_type,
                                "data": base64_image,
                            },
                        },
                    ],
                }
            ],
        )
        return response.content[0].text

    @retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
    def _generate_gpt4o(self, image_path: str) -> str:
        base64_image = encode_image(image_path)
        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Describe this image in one concise sentence, no longer than 20 words.",
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                }
            ],
            max_tokens=100,
        )
        return response.choices[0].message.content

# Ghi lại chính cell đã chạy; không lưu giá trị secrets.
from IPython import get_ipython
IMPLEMENTATION_SOURCES = {}
notebook_shell = get_ipython()
if notebook_shell is not None:
    IMPLEMENTATION_SOURCES["legacy_helpers.py"] = notebook_shell.history_manager.input_hist_raw[-1]


In [ ]:
# Cell 5 — Adapter 5 model chép nguyên văn từ VersionedImageDescriptionGenerator.
# Muốn chỉnh tham số hardcode (temperature/retry/device/dtype/NF4...), sửa ngay dưới đây.
class VersionedImageDescriptionGenerator(ImageDescriptionGenerator):
    """Common caption prompt; modern adapters without changing legacy generators."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.model_name = cfg.caption.model_id
        self.metadata = {}
        if cfg.caption.provider == 'openai':
            self.client = OpenAI(api_key=KaggleRuntime.secret('OPENAI_API_KEY'), timeout=cfg.caption.timeout, max_retries=3)
        elif cfg.caption.provider == 'gemini':
            from google.genai import types
            self.client = genai.Client(api_key=KaggleRuntime.secret('GEMINI_API_KEY'),
                                       http_options=types.HttpOptions(timeout=int(cfg.caption.timeout * 1000)))
        elif cfg.caption.provider == 'huggingface':
            self._load_local()
        else:
            raise ValueError(f'Unknown provider: {cfg.caption.provider}')

    def _load_local(self):
        from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
        if not torch.cuda.is_available():
            raise RuntimeError('Local VLM inference requires a Kaggle GPU.')
        torch.manual_seed(self.cfg.experiment.seed)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        options = dict(device_map={'': 0}, torch_dtype=dtype, low_cpu_mem_usage=True, attn_implementation='eager',
                       revision=self.cfg.caption.revision, trust_remote_code=False)
        if self.cfg.caption.quantization == 'nf4':
            options['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
        elif self.cfg.caption.quantization != 'none':
            raise ValueError('Use quantization=nf4 or none.')
        self.processor = AutoProcessor.from_pretrained(self.model_name, revision=self.cfg.caption.revision, trust_remote_code=False)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_name, **options).eval()
        self.metadata = {'resolved_revision': getattr(self.model.config, '_commit_hash', None),
                         'dtype': str(dtype), 'quantization': self.cfg.caption.quantization,
                         'gpu': torch.cuda.get_device_name(0)}

    @retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(3), reraise=True)
    def generate_description(self, image_path):
        cfg = self.cfg.caption
        self.last_response = {}
        if cfg.provider == 'openai':
            response = self.client.responses.create(
                model=self.model_name, store=False, reasoning={'effort': cfg.reasoning_effort},
                max_output_tokens=cfg.max_output_tokens,
                input=[{'role': 'user', 'content': [
                    {'type': 'input_text', 'text': cfg.prompt},
                    {'type': 'input_image', 'image_url': f'data:image/png;base64,{encode_image(image_path)}', 'detail': 'high'},
                ]}])
            if response.status != 'completed':
                raise ValueError(f'Incomplete OpenAI response: {response.status}')
            text = response.output_text
            self.last_response = {'response_id': response.id, 'model': response.model,
                                  'usage': response.usage.model_dump(mode='json') if response.usage else None}
        elif cfg.provider == 'gemini':
            from google.genai import types
            with open(image_path, 'rb') as stream:
                part = types.Part.from_bytes(data=stream.read(), mime_type='image/png')
            response = self.client.models.generate_content(model=self.model_name, contents=[cfg.prompt, part],
                config=types.GenerateContentConfig(temperature=0, max_output_tokens=cfg.max_output_tokens,
                                                   thinking_config=types.ThinkingConfig(thinking_budget=0)))
            if not response.candidates or 'STOP' not in str(response.candidates[0].finish_reason):
                raise ValueError('Gemini output blocked or truncated.')
            text = response.text
            self.last_response = {'model': getattr(response, 'model_version', self.model_name),
                                  'usage': response.usage_metadata.model_dump(mode='json') if response.usage_metadata else None}
        else:
            messages = [{'role': 'user', 'content': [{'type': 'image', 'url': os.path.abspath(image_path)},
                                                    {'type': 'text', 'text': cfg.prompt}]}]
            inputs = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                                         return_dict=True, return_tensors='pt')
            inputs = inputs.to(self.model.device, dtype=self.model.dtype)
            with torch.inference_mode():
                tokens = self.model.generate(**inputs, do_sample=False, max_new_tokens=cfg.max_new_tokens)
            generated = tokens[0, inputs['input_ids'].shape[1]:]
            if len(generated) >= cfg.max_new_tokens:
                raise ValueError('Local caption reached max_new_tokens; raise the limit in a new config version.')
            text = self.processor.decode(generated, skip_special_tokens=True)
        if not text or not text.strip():
            raise ValueError('Empty caption; refusal/failure is not a valid description.')
        return text.strip()

# Snapshot actual notebook code, bao gồm các sửa đổi bạn thực hiện trên Kaggle.
if notebook_shell is not None:
    IMPLEMENTATION_SOURCES["caption_generator.py"] = notebook_shell.history_manager.input_hist_raw[-1]


In [ ]:
# Cell 6 — Nối manifest của test_attack_v1; không thay đổi hàm sinh caption.
from PIL import Image

samples = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
if not samples:
    raise ValueError("Manifest rỗng. Chạy attack với RUN_ATTACK=True trước.")
ROLES = ("source", "target", "adversarial")
seen_indices = set()
validated_samples = []
for sample in samples:
    if type(sample.get("index")) is not int or sample["index"] in seen_indices:
        raise ValueError("Manifest thiếu index hợp lệ hoặc trùng index.")
    seen_indices.add(sample["index"])
    images, image_sha256 = {}, {}
    for role in ROLES:
        relative = sample.get(role)
        if not isinstance(relative, str) or Path(relative).is_absolute():
            raise ValueError(f"Invalid image path: index={sample['index']}, role={role}")
        path = (INPUT_ROOT / relative).resolve()
        if not path.is_relative_to(INPUT_ROOT) or not path.is_file():
            raise ValueError(f"Unsafe or missing image: {path}")
        # Generator gốc dùng MIME image/png; test_attack_v1 xuất PNG lossless.
        with Image.open(path) as image:
            if image.format != "PNG":
                raise ValueError(f"Input cần là PNG từ test_attack_v1: {path}")
            image.verify()
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        images[role] = relative
        image_sha256[role] = digest.hexdigest()
    validated_samples.append({
        **sample, "sample_id": f"{sample['index']:05d}",
        "images": images, "image_sha256": image_sha256,
    })
selected_samples = validated_samples if NUM_SAMPLES is None else validated_samples[:NUM_SAMPLES]
print(f"Validated {len(validated_samples)} samples; selected {len(selected_samples)}.")
print("First sample:", selected_samples[0]["images"])


In [ ]:
# Cell 7 — Chạy class Cell 5; log cấu hình/code thực thi và ghi từng caption record.
from datetime import datetime, timezone
import shutil
from tqdm.auto import tqdm

RUN_DIR = None
if RUN_CAPTION:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    RUN_DIR = OUTPUT_DIR / f"{cfg.experiment.name}_{stamp}"
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    config_data = OmegaConf.to_container(cfg, resolve=True)
    implementation_hashes = {
        filename: hashlib.sha256(source.encode("utf-8")).hexdigest()
        for filename, source in IMPLEMENTATION_SOURCES.items()
    }
    fingerprint_data = {
        "config": config_data, "packages": PACKAGES,
        "implementation_sha256": implementation_hashes,
    }
    fingerprint = hashlib.sha256(json.dumps(fingerprint_data, sort_keys=True).encode()).hexdigest()
    (RUN_DIR / "config_resolved.yaml").write_text(OmegaConf.to_yaml(cfg, resolve=True), encoding="utf-8")
    (RUN_DIR / "config_resolved.json").write_text(json.dumps(config_data, indent=2, ensure_ascii=False), encoding="utf-8")
    reference_hashes = {'src/FOA-Attack/blackbox_text_generation.py': '0eee576410ffd88e44e5f512f3a16e2b04ead0a1ce4f047f3ca545842f737856', 'src/FOA-Attack/utils.py': '110f2a3417a3f0ea195f22d6352849a6780e3016f024e1bc205ba028ada3f58c'}
    run_config = {
        "notebook": "our_test_generation_v1.1.ipynb", "self_contained": True,
        "reference_source_sha256": reference_hashes,
        "model": MODEL, "caption_model": cfg.caption.model_id,
        "caption_config_sha256": fingerprint, "fingerprint_inputs": fingerprint_data,
        "manifest": str(MANIFEST_PATH), "manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
        "input_samples": len(validated_samples), "selected_samples": len(selected_samples),
        "num_samples": NUM_SAMPLES, "roles": list(ROLES),
        "device": "cuda:0" if LOCAL_MODEL else "API",
        "python": sys.version, "torch": torch.__version__ if LOCAL_MODEL else None,
        "implementation_snapshot_available": bool(IMPLEMENTATION_SOURCES),
    }
    (RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")
    shutil.copy2(MANIFEST_PATH, RUN_DIR / "manifest.jsonl")
    if (INPUT_ROOT / "run_config.json").is_file():
        shutil.copy2(INPUT_ROOT / "run_config.json", RUN_DIR / "attack_run_config.json")
    if IMPLEMENTATION_SOURCES:
        (RUN_DIR / "code").mkdir()
        for filename, source in IMPLEMENTATION_SOURCES.items():
            (RUN_DIR / "code" / filename).write_text(source, encoding="utf-8")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
    (RUN_DIR / "requirements-resolved.txt").write_text(freeze.stdout, encoding="utf-8")
    failures, completed_samples, cache = 0, 0, {}
    status = {"status": "running", "samples": 0, "selected_samples": len(selected_samples), "failed_captions": 0}
    (RUN_DIR / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    try:
        generator = VersionedImageDescriptionGenerator(cfg)
        (RUN_DIR / "model_metadata.json").write_text(
            json.dumps({"model_id": generator.model_name, **generator.metadata}, indent=2), encoding="utf-8")
        with (RUN_DIR / "captions.jsonl").open("w", encoding="utf-8") as output:
            for sample in tqdm(selected_samples, desc=generator.model_name):
                row = {
                    **sample, "caption_experiment": cfg.experiment.name,
                    "caption_model": generator.model_name, "caption_config_sha256": fingerprint,
                    "captions": {}, "caption_responses": {}, "errors": {},
                }
                for role in ROLES:
                    checksum = sample["image_sha256"][role]
                    path = INPUT_ROOT / sample["images"][role]
                    try:
                        if hashlib.sha256(path.read_bytes()).hexdigest() != checksum:
                            raise ValueError("Image changed since input validation.")
                        if checksum not in cache:
                            caption = generator.generate_description(str(path))
                            cache[checksum] = (caption, generator.last_response)
                        row["captions"][role], row["caption_responses"][role] = cache[checksum]
                    except Exception as error:
                        row["errors"][role] = type(error).__name__
                        failures += 1
                row["status"] = "ok" if not row["errors"] else "error"
                if row["status"] == "ok":
                    assert set(row["captions"]) == set(ROLES)
                output.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
                output.flush()
                completed_samples += 1
        status["status"] = "complete" if not failures else "partial"
    except BaseException as error:
        status["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
        status["error_type"] = type(error).__name__
        print("Run stopped:", status["error_type"], "| Export artifacts in Cell 8.")
    finally:
        status.update(samples=completed_samples, failed_captions=failures, unique_captions=len(cache))
        (RUN_DIR / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("Output:", RUN_DIR, "| Status:", status["status"])
else:
    print("Input/config validated. Đặt RUN_CAPTION=True tại Cell 1 rồi chạy lại.")


In [ ]:
# Cell 8 — Preview ba caption và export ZIP, kể cả run dở dang.
from IPython.display import display, FileLink
import matplotlib.pyplot as plt

if RUN_DIR is not None and RUN_DIR.is_dir():
    captions_path = RUN_DIR / "captions.jsonl"
    caption_rows = [json.loads(line) for line in captions_path.read_text(encoding="utf-8").splitlines() if line.strip()] if captions_path.is_file() else []
    if caption_rows:
        preview = caption_rows[0]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for axis, role in zip(axes, ROLES):
            with Image.open(INPUT_ROOT / preview["images"][role]) as image:
                axis.imshow(image)
            axis.set_title(role)
            axis.axis("off")
        plt.show()
        for role in ROLES:
            print(f"{role}: {preview['captions'].get(role, '[error: ' + preview['errors'].get(role, 'missing') + ']')}")
    ZIP_PATH = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
    print("Status:", json.loads((RUN_DIR / "status.json").read_text(encoding="utf-8")))
    print("Input cho GPT evaluation:", captions_path)
    display(FileLink(str(ZIP_PATH)))
else:
    print("Chưa chạy caption. Đặt RUN_CAPTION=True rồi chạy lại các cell.")
